In [ ]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)

# 1. Setup Checkpoints (Switching to V3-Base as requested)
checkpoint = "microsoft/deberta-v3-base"
dataset_name = "imdb"

# 2. Device Check
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {DEVICE}")

# 3. Load Dataset and Tokenizer
# Using standard load_dataset to bypass custom tool issues
dataset = load_dataset(dataset_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(examples):
    # DeBERTa-v3 is powerful, but max_length 512 is the standard limit
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Data collator handles dynamic padding for batches
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Load Model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.config.problem_type = "single_label_classification"
model.to(DEVICE)

# 5. Metrics (Accuracy)
metric = evaluate.load("accuracy")


/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training on: cuda


/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map:  44%|████▍     | 11000/25000 [00:03<00:03, 3599.94 examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6. Training Arguments
training_args = TrainingArguments(
    output_dir="./deberta-imdb-finetuned",
    learning_rate=2e-5,            # Standard for DeBERTa
    per_device_train_batch_size=8, # Adjust based on your VRAM (8-16 is good)
    per_device_eval_batch_size=8,
    num_train_epochs=1,            # IMDB usually converges in 1-2 epochs
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                     # Highly recommended for speed/memory
    logging_steps=100,
)

# 7. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 8. Start Training
print("Starting training...")
trainer.train()

# 9. Save the Final Model
trainer.save_model("./final_deberta_imdb_model")
print("Done!")

/tmp/ipykernel_1032663/1849832471.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

Starting training...


wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice: